---
## 9. RAG — Retrieval Augmented Generation

RAG = **Retrieve** relevant docs from your knowledge base + **Augment** the LLM prompt with those docs + **Generate** an answer.

```
User Question
     ↓
Retriever (FAISS / Vector DB)
     ↓
Relevant Text Chunks
     ↓
LLM (Chat Model)
     ↓
Final Answer
```

## 📌 What is RAG?
RAG = Retrieval Augmented Generation

It means:
1. We first find relevant documents (retrieval)
2. Then we send them to LLM (generation)

👉 LLM answers using your data

In [16]:
!pip install langchain langchain-community langchain-groq faiss-cpu sentence-transformers

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\Nidhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 🔑 Setup API Key

In [17]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("API Loaded!")

API Loaded!


## 🧠 Step 1: Embeddings

In [18]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings ready!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1305.72it/s]


Embeddings ready!


## 📄 Step 2: Load Data

In [19]:
text = """LangChain is a framework for building LLM applications.
It was created by Harrison Chase.
It helps in building AI apps quickly."""

from langchain_community.document_loaders import TextLoader
with open("sample.txt", "w") as f:
    f.write(text)

loader = TextLoader("sample.txt")
docs = loader.load()

print(docs[0].page_content)

LangChain is a framework for building LLM applications.
It was created by Harrison Chase.
It helps in building AI apps quickly.


## ✂️ Step 3: Split Text

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter # break big text into small chunks


splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = splitter.split_documents(docs)

print(len(chunks))

2


## 📦 Step 4: Vector Store (FAISS)

In [21]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)
print("Vector DB created!")

Vector DB created!


## 🔍 Step 5: Retriever

In [22]:
retriever = vectorstore.as_retriever()

docs = retriever.invoke("Who created LangChain?")
for d in docs:
    print(d.page_content)

LangChain is a framework for building LLM applications.
It was created by Harrison Chase.
It helps in building AI apps quickly.


## 🤖 Step 6: LLM

In [23]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant")

print("LLM ready!")

LLM ready!


## 🧠 Step 7: RAG Chain

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only context"),
    ("human", "Context: {context}\n\nQuestion: {question}")
])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG ready!")

RAG ready!


## 🎯 Step 8: Test

In [25]:
question = "Who created LangChain?"
print(rag_chain.invoke(question))

Harrison Chase.
